In [6]:
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from openai import OpenAI

openai_client = OpenAI()


documents = load_faq_data()
index = build_index(documents)




In [7]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [ ]:
instructionss = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [8]:
#instructions that force multiple searches and block irrelevant data

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. Make at least two searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [9]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [52]:
question = "I just discovered the course. Can I join it?"

In [10]:
def make_call(call):
    import json
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [17]:
question = "I just discovered the course. Can I join it?"

In [11]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [12]:
agent_loop(instructions, "Can I still take this class?")

iteration #1...
function_call: search {"query":"Can I still take this class enrollment registration add class late join"}
iteration #2...
function_call: search {"query":"still join course certificate submit project while accepting submissions registration not needed"}
iteration #3...
ASSISTANT:
Yes — you can still join the class. You don’t need a confirmation email, and registration is just to gauge interest before the start date.

If you want a certificate, you need to submit your project while submissions are still open, and certificates are only awarded for the live cohort, not self-paced mode.

Do you want help with anything else about the course?


'Yes — you can still join the class. You don’t need a confirmation email, and registration is just to gauge interest before the start date.\n\nIf you want a certificate, you need to submit your project while submissions are still open, and certificates are only awarded for the live cohort, not self-paced mode.\n\nDo you want help with anything else about the course?'

In [48]:
usage =response.usage
usage.input_tokens, usage.output_tokens

(987, 229)

In [49]:
def calculate_cost(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 1.222
    OUTPUT_PRICE_PER_MILLION = 33.0

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }

In [50]:
calculate_cost(usage.input_tokens, usage.output_tokens)

{'input_cost': 0.001206114, 'output_cost': 0.007557, 'total_cost': 0.008763114}